# tw-airquality-mini — Exploration

Per-station 2025 air quality CSVs in `data/raw/`. Long format: `測站, 日期, 測項, 00..23` (hourly). This notebook focuses on PM2.5.

In [ ]:
import sys as _sys
if _sys.platform == "win32":
    import asyncio as _asyncio
    _asyncio.set_event_loop_policy(_asyncio.WindowsSelectorEventLoopPolicy())
del _sys

import glob
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

plt.rcParams['font.sans-serif'] = ['Microsoft JhengHei', 'Microsoft YaHei', 'PingFang TC', 'Noto Sans CJK TC', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

RAW_DIR = Path('../data/raw')
REPORTS_DIR = Path('../reports')
HOUR_COLS = [f'{h:02d}' for h in range(24)]

In [ ]:
def load_pm25_daily(raw_dir: Path) -> pd.DataFrame:
    """Load all stations, keep PM2.5 rows, melt to long, return daily mean per station."""
    frames = []
    for csv_path in sorted(glob.glob(str(raw_dir / '*.csv'))):
        df = pd.read_csv(csv_path, encoding='utf-8-sig')
        df = df[df['測項'] == 'PM2.5']
        long = df.melt(
            id_vars=['測站', '日期'],
            value_vars=HOUR_COLS,
            var_name='hour',
            value_name='pm25',
        )
        long['pm25'] = pd.to_numeric(long['pm25'], errors='coerce')
        long['date'] = pd.to_datetime(long['日期']).dt.normalize()
        frames.append(long[['測站', 'date', 'hour', 'pm25']])
    all_long = pd.concat(frames, ignore_index=True)
    daily = (
        all_long.groupby(['測站', 'date'], as_index=False)['pm25']
        .mean()
        .rename(columns={'測站': 'station'})
    )
    return daily

In [ ]:
!pwd
pm25_daily = load_pm25_daily(RAW_DIR)

In [ ]:
pm25_daily = load_pm25_daily(RAW_DIR)
print(f'Stations: {pm25_daily["station"].nunique()}, days: {pm25_daily["date"].nunique()}, rows: {len(pm25_daily)}')
pm25_daily.head()

# Summary stats per station — annual mean / median / max PM2.5
summary = (
    pm25_daily.groupby('station')['pm25']
    .agg(['mean', 'median', 'std', 'min', 'max', 'count'])
    .round(2)
    .sort_values('mean', ascending=False)
)
print('Top 10 stations by annual mean PM2.5 (μg/m³):')
print(summary.head(10).to_string())
print()
print(f'Available stations ({len(summary)}):')
print('  ' + ', '.join(summary.index.tolist()))

# Interactive prompt; falls back to the top-mean station when stdin is
# unavailable (nbconvert --execute, CI, etc.).
try:
    district = input('\nEnter district (測站) to plot: ').strip() or summary.index[0]
except Exception:
    district = summary.index[0]
if district not in summary.index:
    raise ValueError(f'Unknown district: {district!r}')
print(f'Selected: {district}')

In [ ]:
series = (
    pm25_daily[pm25_daily['station'] == district]
    .sort_values('date')
    .set_index('date')['pm25']
)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(series.index, series.values, linewidth=1)
ax.set_title(f'Daily mean PM2.5 — {district} (2025)')
ax.set_xlabel('Date')
ax.set_ylabel('PM2.5 (μg/m³)')
ax.grid(True, alpha=0.3)
fig.autofmt_xdate()
plt.show()

In [ ]:
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
out_path = REPORTS_DIR / 'pm25_demo.png'
fig.savefig(out_path, dpi=150, bbox_inches='tight')
print(f'Saved: {out_path.resolve()}')